# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |RM024_Economic_activity_status_by_age |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |RM024_Economic_activity_status_by_age |
| Output File |economic_activity_status.json |
| Notebook Author | Peter |
| Last Updated | 2026-07-13|

## 1. Import

In [2]:
# Import libraries

import pandas as pd
import numpy as np
import json

In [3]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [4]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [5]:
# File paths

input_file = "RM024_Economic_activity_status_by_age.xlsx"
output_file = "economic_activity_status.json"

raw_data_path = Path("../../data/demographic/raw/economic_activity_status/")
output_data_path = Path("../../data/demographic/processed/economic_activity_status/")

## 3. Load Dataset

In [6]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Economic activity status (7 categories) Code,Economic activity status (7 categories),Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0,Aged under 1 year,918
1,E06000001,Hartlepool,-8,Does not apply,1,Aged 1 year,954
2,E06000001,Hartlepool,-8,Does not apply,2,Aged 2 years,1046
3,E06000001,Hartlepool,-8,Does not apply,3,Aged 3 years,988
4,E06000001,Hartlepool,-8,Does not apply,4,Aged 4 years,1077


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [7]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198660 entries, 0 to 198659
Data columns (total 7 columns):
 #   Column                                        Non-Null Count   Dtype 
---  ------                                        --------------   ----- 
 0   Lower tier local authorities Code             198660 non-null  object
 1   Lower tier local authorities                  198660 non-null  object
 2   Economic activity status (7 categories) Code  198660 non-null  int64 
 3   Economic activity status (7 categories)       198660 non-null  object
 4   Age (86 categories) Code                      198660 non-null  int64 
 5   Age (86 categories)                           198660 non-null  object
 6   Observation                                   198660 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 10.6+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Economic activity status (7 categories) Code',
       'Economic activity status (7 categories)', 'Age (86 categories) Code',
       'Age (86 categories)', 'Observation'],
      dtype='object')

In [8]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Economic activity status (7 categories) Code,Economic activity status (7 categories),Age (86 categories) Code,Age (86 categories),Observation
count,198660,198660,198660.000000,198660,198660.000000,198660,198660.000000
unique,330,330,NaN,7,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Does not apply,NaN,Aged under 1 year,NaN
freq,602,602,NaN,28380,NaN,2310,NaN
mean,NaN,NaN,1.857143,NaN,42.500000,NaN,299.985518
std,NaN,NaN,4.323653,NaN,24.824446,NaN,779.335262
min,NaN,NaN,-8.000000,NaN,0.000000,NaN,0.000000
25%,NaN,NaN,1.000000,NaN,21.000000,NaN,0.000000
50%,NaN,NaN,3.000000,NaN,42.500000,NaN,4.000000
75%,NaN,NaN,5.000000,NaN,64.000000,NaN,142.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [9]:
# Cleaning code
selected_columns = ['Lower tier local authorities',
                    'Economic activity status (7 categories)',
                    'Age (86 categories) Code',
                    'Observation']
df_processed = df_processed[selected_columns]

df_processed = df_processed.rename(columns={
    'Economic activity status (7 categories)': 'Economic Activity Status',
    'Age (86 categories) Code': 'Age',
    'Observation': 'Economic Count'
})

## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [10]:
# Dataset-specific processing
from ontology.FRAMEWORK_AGE_GROUP import (
    FRAMEWORK_AGE_GROUP,
    AGE_ORDER
)

In [11]:
def get_economic_activity_distribution(local_authority):
    filtered_data = df_processed[
        df_processed['Lower tier local authorities'] == local_authority].copy()

    filtered_data['Age'] = filtered_data['Age'].apply(FRAMEWORK_AGE_GROUP)

    result = (filtered_data.groupby(['Lower tier local authorities', 
                                     'Economic Activity Status',
                                     'Age'],as_index=False)['Economic Count'].sum())

    result['Age'] = pd.Categorical(result['Age'], categories=AGE_ORDER, ordered=True)

    result = result.sort_values('Age').reset_index(drop=True)

    return result

In [12]:
from ontology.STUDY_AREA import STUDY_AREA

In [13]:
get_economic_activity_distribution(STUDY_AREA)

,Lower tier local authorities,Economic Activity Status,Age,Economic Count
0,Manchester,Does not apply,0-4,34378
1,Manchester,Economically active and a full-time student: I...,0-4,0
2,Manchester,Economically inactive (excluding full-time stu...,0-4,0
3,Manchester,Economically active and a full-time student: U...,0-4,0
4,Manchester,Economically inactive and a full-time student,0-4,0
5,Manchester,Economically active (excluding full-time stude...,0-4,0
6,Manchester,Economically active (excluding full-time stude...,0-4,0
7,Manchester,Economically active and a full-time student: I...,5-11,0
8,Manchester,Does not apply,5-11,51349
9,Manchester,Economically active (excluding full-time stude...,5-11,0


In [14]:
df_processed["Economic Activity Status"].unique()

array(['Does not apply',
       'Economically active (excluding full-time students): In employment',
       'Economically active (excluding full-time students): Unemployed: Seeking work or waiting to start a job already obtained: Available to start working within 2 weeks',
       'Economically active and a full-time student: In employment',
       'Economically active and a full-time student: Unemployed: Seeking work or waiting to start a job already obtained: Available to start working within 2 weeks',
       'Economically inactive (excluding full-time students)',
       'Economically inactive and a full-time student'], dtype=object)

## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [ ]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198660 entries, 0 to 198659
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Lower tier local authorities  198660 non-null  object
 1   Economic Activity Status      198660 non-null  object
 2   Age                           198660 non-null  int64 
 3   Economic Count                198660 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 6.1+ MB


Lower tier local authorities    0
Economic Activity Status        0
Age                             0
Economic Count                  0
dtype: int64

## 8. Export Processed Dataset

In [ ]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

# df_processed.to_json(
#     output_data_path / output_file,
#     orient="records",
#     indent=4
# )

economic_activity_status_df = get_economic_activity_distribution("Manchester")
economic_activity_status_df.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported